### Preamble

In [ ]:
# install packages if needed
# install.packages("tidyverse")
# install.packages("WDI")

# load libraries
library("tidyverse")
library("WDI")

options(repr.plot.width = 12, repr.plot.height = 8)

In [ ]:
# directory
setwd('/Users/tasia/Desktop/Data Analysis/Assessment')

### Load and set up GDP and Unemployment data

In [ ]:
# load data
df_gdp <- read_csv("cross_sectional_gdp_2023.csv")
df_unp <- read_csv("cross_sectional_unemployment_2023.csv")

# eyeball
head(df_gdp)
head(df_unp)

In [ ]:
# convert columns to numeric
df_gdp$`2023 [YR2023]` <- as.numeric(na_if(df_gdp$`2023 [YR2023]`, ".."))
df_unp$`2023 [YR2023]` <- as.numeric(na_if(df_unp$`2023 [YR2023]`, ".."))

# eyeball
head(df_gdp)
head(df_unp)

In [ ]:
# remove rows with missing country codes
df_gdp <- filter(df_gdp, !is.na(`Country Code`))
df_unp <- filter(df_unp, !is.na(`Country Code`))

# merge dataframes (inner join!)
df <- inner_join(df_gdp,
                 df_unp,
                 by = "Country Code")

# eyeball
head(df)

In [ ]:
# rename columns
df <- rename(df,
             gdp = `2023 [YR2023].x`,
             unemployment = `2023 [YR2023].y`)

# eyeball
head(df)

In [ ]:
# load country information
countries <- WDI_data$country

# remove World Bank aggregates
countries <- filter(countries, region != "Aggregates")
df <- filter(df, `Country Code` %in% countries$iso3c)

# remove missing values
df <- drop_na(df, gdp, unemployment)

# add region information
df <- left_join(df,
                select(countries, iso3c, region),
                by = c("Country Code" = "iso3c"))

# eyeball
head(df)

### Run a regression

In [ ]:
# regress unemployment on GDP per capita
model1 <- lm(unemployment ~ gdp, data = df)
summary(model1)

In [ ]:
# regress unemployment on logged GDP per capita
model2 <- lm(unemployment ~ log(gdp), data = df)
summary(model2)

In [ ]:
# regress unemployment on GDP per capita and GDP per capita squared
model3 <- lm(unemployment ~ gdp + I(gdp^2), data = df)
summary(model3)

In [ ]:
# regress unemployment on GDP per capita and region
model4 <- lm(unemployment ~ gdp + region, data = df)
summary(model4)